# Chequeo de la distancia estimada fuera de fold

Complemento de las semanas 12-13. La distancia estimada (mediana histórica de `trip_distance` por par de zonas) es la feature más importante del modelo final: ~75 % del *Gain* en el notebook 06.

Se aprende solo con train, así que validación y test no tienen fuga. Pero **dentro de train** cada viaje entra en la mediana de su propio par, y en pares con pocos viajes la feature se acerca a la distancia real del viaje (con un solo viaje, es exactamente su distancia). El TargetEncoder evita este problema con validación cruzada interna (`cv=5`); la distancia no. Aquí se calcula la distancia de train **fuera de fold** (5 folds) y se compara contra el modelo tal cual.

**Protocolo:** el mismo de los notebooks 04 y 04b. Train con el 50 % de enero-abril, validación con mayo (la misma muestra del 20 % de los notebooks 02-04b) y la configuración de `outputs/config_modelo_final.json`. **Junio no se usa**: el modelo final (L) y su resultado en test no cambian por este notebook.

Tarda cerca de 15 minutos y necesita unos 8 GB de RAM.

In [1]:
import sys
sys.path.append('..')

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold

import src.features as features
features.RUTA_ZONAS = Path('../data/Taxi_Zone_Lookup.csv')

from src.data import cargar_mes, limpiar_mes
from src.features import construir_preprocesador, preparar_xy, ExtractorFeaturesNYC
from src import tuning

SEED = 42
np.random.seed(SEED)
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

with open('../outputs/config_modelo_final.json', encoding='utf-8') as f:
    config = json.load(f)
FRAC_TRAIN = 0.50
N_ARBOLES = config['n_estimators']
print(f"Configuración final: {config['params']} | {N_ARBOLES} árboles | train {FRAC_TRAIN:.0%}")

Configuración final: {'objective': 'regression_l1', 'learning_rate': 0.05, 'num_leaves': 281, 'min_child_samples': 30, 'subsample': 0.78, 'colsample_bytree': 0.685, 'reg_alpha': 5.094, 'reg_lambda': 0.3762} | 1710 árboles | train 50%


## 1. Datos

Igual que en el notebook 04b: cada mes se carga y limpia con `src/data.py`, sin `cargar_todo` (que reescribe `reports/limpieza_reglas.csv`), y se muestrea por mes con `SEED = 42`.

In [2]:
def mes_limpio(month):
    df, _ = limpiar_mes(cargar_mes(2025, month), 2025, month)
    return df

train = pd.concat(
    [mes_limpio(m).sample(frac=FRAC_TRAIN, random_state=SEED) for m in (1, 2, 3, 4)],
    ignore_index=True,
)
val = mes_limpio(5).sample(frac=0.20, random_state=SEED)

X_tr, y_tr_log = preparar_xy(train, usar_log_target=True)
X_val, _ = preparar_xy(val)
y_val = val['trip_duration_min'].to_numpy(dtype=float)
dist_real_tr = train['trip_distance'].to_numpy(dtype=float)
dist_real_val = val['trip_distance'].to_numpy(dtype=float)
del train, val

t0 = time.time()
prep = construir_preprocesador(escalar_numericas=False)
Xt_tr = prep.fit_transform(X_tr, y_tr_log).astype(np.float32)
Xt_val = prep.transform(X_val).astype(np.float32)
nombres = list(prep.named_steps['codificador'].get_feature_names_out())
print(f"Train (ene-abr, {FRAC_TRAIN:.0%}): {len(X_tr):,} | Val (mayo, 20 %): {len(X_val):,} | "
      f"{len(nombres)} features | preprocesamiento {time.time() - t0:.0f} s")

Train (ene-abr, 50%): 7,181,104 | Val (mayo, 20 %): 855,412 | 41 features | preprocesamiento 48 s


## 2. Función de entrenamiento

Misma configuración que el modelo K del notebook 04b (y el modelo final L del notebook 05, salvo que L se entrena con enero-mayo).

In [3]:
def entrenar(nombre, X_train, X_validacion, resultados):
    t0 = time.time()
    modelo = lgb.LGBMRegressor(n_estimators=N_ARBOLES, **tuning.PARAMS_FIJOS, **config['params'])
    modelo.fit(X_train, y_tr_log)
    m = tuning.metricas(y_val, np.expm1(modelo.predict(X_validacion)))
    segundos = round(time.time() - t0, 1)
    print(f"{nombre:<45} MAE = {m['MAE']:.4f} | RMSLE = {m['RMSLE']:.4f} | {segundos:.0f} s", flush=True)
    resultados.append({'modelo': nombre, **m, 'segundos': segundos})
    return modelo


resultados = []

## 3. Control: modelo K tal cual

Debe reproducir el MAE de mayo guardado en `config_modelo_final.json` (3.918 min). Si no lo reproduce, las comparaciones siguientes no son válidas.

In [4]:
entrenar('K: configuración final (control)', Xt_tr, Xt_val, resultados)

if FRAC_TRAIN == config['sample_frac']:
    assert abs(resultados[0]['MAE'] - config['val_mayo']['MAE']) < 1e-3, "No se reproduce el modelo K del notebook 04b"
    print(f"Control OK: reproduce el MAE de mayo del notebook 04b ({config['val_mayo']['MAE']:.4f})")

K: configuración final (control)              MAE = 3.9180 | RMSLE = 0.3125 | 351 s


Control OK: reproduce el MAE de mayo del notebook 04b (3.9180)


## 4. Distancia estimada fuera de fold en train

Para cada uno de 5 folds de train (`KFold` con mezcla, como el TargetEncoder), se ajusta `ExtractorFeaturesNYC` con los otros 4 y se calcula la distancia estimada de ese fold. Así ningún viaje de train entra en la mediana que se usa para él. Validación no cambia: sigue usando las medianas de todo train, como en producción.

Solo se reemplazan las dos columnas de distancia (`distancia_estimada_mi` y `log_distancia_estimada`); el resto de las features no depende de ellas.

In [5]:
i_dist = nombres.index('num__distancia_estimada_mi')
i_log = nombres.index('num__log_distancia_estimada')

t0 = time.time()
Xt_tr_oof = Xt_tr.copy()
for idx_fit, idx_fold in KFold(n_splits=5, shuffle=True, random_state=SEED).split(X_tr):
    extractor = ExtractorFeaturesNYC(ruta_zonas=features.RUTA_ZONAS).fit(X_tr.iloc[idx_fit])
    f = extractor.transform(X_tr.iloc[idx_fold])
    Xt_tr_oof[idx_fold, i_dist] = f['distancia_estimada_mi'].to_numpy(dtype=np.float32)
    Xt_tr_oof[idx_fold, i_log] = f['log_distancia_estimada'].to_numpy(dtype=np.float32)
print(f"Distancia fuera de fold calculada en {time.time() - t0:.0f} s")

Distancia fuera de fold calculada en 26 s


**Diagnóstico antes de entrenar:** si la versión dentro de fold filtra la distancia real, en train debería parecerse más a `trip_distance` que en validación. Se compara la correlación entre la feature y la distancia real del viaje en cada caso, y cuánto cambia la feature en los viajes de pares poco frecuentes.

In [6]:
dentro = Xt_tr[:, i_dist]
fuera = Xt_tr_oof[:, i_dist]
viajes_por_par = X_tr.groupby(['PULocationID', 'DOLocationID'])['PULocationID'].transform('size').to_numpy()

diagnostico = pd.DataFrame([
    {'conjunto': 'train, dentro de fold', 'corr_con_trip_distance': np.corrcoef(dentro, dist_real_tr)[0, 1]},
    {'conjunto': 'train, fuera de fold', 'corr_con_trip_distance': np.corrcoef(fuera, dist_real_tr)[0, 1]},
    {'conjunto': 'validación (mayo)', 'corr_con_trip_distance': np.corrcoef(Xt_val[:, i_dist], dist_real_val)[0, 1]},
])
raros = viajes_por_par < 5
print(f"Viajes de train en pares con < 5 viajes: {raros.sum():,} ({raros.mean():.2%})")
print(f"Cambio medio |dentro - fuera| (millas): todos = {np.abs(dentro - fuera).mean():.3f} | "
      f"pares con < 5 viajes = {np.abs(dentro - fuera)[raros].mean():.3f}")
diagnostico.round(4)

Viajes de train en pares con < 5 viajes: 36,172 (0.50%)
Cambio medio |dentro - fuera| (millas): todos = 0.029 | pares con < 5 viajes = 3.133


,conjunto,corr_con_trip_distance
0,"train, dentro de fold",0.9738
1,"train, fuera de fold",0.9675
2,validación (mayo),0.9639


In [7]:
entrenar('K con distancia fuera de fold', Xt_tr_oof, Xt_val, resultados)
del Xt_tr_oof

K con distancia fuera de fold                 MAE = 3.9081 | RMSLE = 0.3114 | 355 s


## 5. Resumen

In [8]:
tabla = pd.DataFrame(resultados)
mae_control = tabla.loc[0, 'MAE']
tabla['dif_vs_control_min'] = (tabla['MAE'] - mae_control).round(4)
tabla

,modelo,MAE,RMSLE,segundos,dif_vs_control_min
0,K: configuración final (control),3.918025,0.312452,351.0,0.0000
1,K con distancia fuera de fold,3.908102,0.311419,355.3,-0.0099


In [9]:
Path('../outputs').mkdir(exist_ok=True)
with open('../outputs/metrics_distancia_oof.json', 'w', encoding='utf-8') as f:
    json.dump({'resultados_mayo': resultados, 'diagnostico_distancia': diagnostico.to_dict(orient='records')},
              f, indent=2, ensure_ascii=False)
print("Guardado en ../outputs/metrics_distancia_oof.json")

Guardado en ../outputs/metrics_distancia_oof.json


## 6. Lectura de resultados

| Modelo (validación: mayo) | MAE (min) | RMSLE |
|---|---|---|
| K: configuración final (control) | 3.9180 | 0.3125 |
| K con distancia fuera de fold | **3.9081** | **0.3114** |

**La fuga existe, pero es chica y está concentrada.** Solo el 0.50 % de los viajes de train pertenece a pares con menos de 5 viajes. En ellos, calcular la distancia fuera de fold la cambia en 3.13 millas en promedio; en el conjunto completo, en apenas 0.03 millas. La correlación entre la feature y la distancia real lo confirma: dentro de fold es 0.974 en train, fuera de fold baja a 0.968, más cerca del 0.964 de validación. La versión actual hace que la feature parezca algo más precisa en train de lo que realmente es en datos nuevos.

**Corregirla mejora levemente el modelo.** El MAE de mayo baja 0.010 min (−0.25 %) y el RMSLE de 0.3125 a 0.3114. La diferencia es chica, pero unas 3 veces mayor que la variación por muestreo medida en el notebook 04b (0.003 min) y va en la dirección esperada: con la fuga, el modelo confía de más en la distancia en rutas raras.

**Qué significa para el modelo final.** El resultado en test (MAE = 3.694 min en junio, notebook 05) sigue siendo honesto: las distancias de junio se calcularon solo con medianas de enero-mayo, sin ningún viaje de junio. La fuga afectaba solo a cómo aprende el modelo dentro de train, no a la evaluación. Como el efecto en validación es de 0.01 min, no cambia ninguna conclusión del proyecto ni el orden de los modelos.

**Decisión:** el modelo final (L) se mantiene como resultado oficial, porque cambiarlo después de haber visto junio sería usar el test para decidir. En el informe se documenta como chequeo de robustez y se deja como mejora recomendada para una versión futura: calcular la distancia fuera de fold, igual que el TargetEncoder.

Las métricas están en `outputs/metrics_distancia_oof.json`.